# NLP Sentiment & Emotion Classification — Classical LSTM Pipeline

End-to-end LSTM-based emotion classifier following the standard NLP pipeline:

1. **Load data** — `dair-ai/emotion` (6-way: Sadness, Joy, Love, Anger, Fear, Surprise)
2. **Preprocessing** — lowercase → strip punctuation/digits → remove stopwords → lemmatize
3. **EDA** — class-distribution chart + per-emotion WordClouds
4. **Tokenize + pad** — Keras `Tokenizer` → `texts_to_sequences` → `pad_sequences`
5. **Encode labels** — integer → one-hot for softmax
6. **Model** — `Embedding → BiLSTM → Dropout → Dense → Dropout → Dense(softmax)`
7. **Train** — Adam + categorical cross-entropy, ~15 epochs with EarlyStopping
8. **Evaluate** — test accuracy, per-class precision/recall/F1, confusion matrix
9. **Inference** — `predict_emotion("text here")` helper

> Runtime → Change runtime type → **GPU** (optional; LSTM is fast on CPU too).


## 1 — Install & download NLTK assets

In [ ]:
!pip install -q -U "datasets>=2.16" nltk wordcloud scikit-learn matplotlib seaborn

import nltk
for pkg in ['stopwords', 'wordnet', 'omw-1.4', 'punkt', 'punkt_tab']:
    try:
        nltk.download(pkg, quiet=True)
    except Exception as e:
        print(f"NLTK '{pkg}': {e}")
print("NLTK ready.")

## 2 — Imports & seed

In [ ]:
import os, re, string, random, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from datasets import load_dataset
from sklearn.metrics import (classification_report, confusion_matrix,
                              accuracy_score, f1_score)

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Embedding, Bidirectional, LSTM, Dense,
                                      Dropout, SpatialDropout1D)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.utils import to_categorical

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print("TF:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU') or "CPU only (fine for LSTM)")

## 3 — Step 1: Load data

In [ ]:
try:
    ds = load_dataset("dair-ai/emotion", trust_remote_code=True)
except TypeError:
    ds = load_dataset("dair-ai/emotion")

train_df = pd.DataFrame(ds['train'])
val_df   = pd.DataFrame(ds['validation'])
test_df  = pd.DataFrame(ds['test'])

EMOTION_LABELS = ["Sadness", "Joy", "Love", "Anger", "Fear", "Surprise"]
NUM_CLASSES    = len(EMOTION_LABELS)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
train_df.head()

## 4 — Step 2: Text preprocessing
Lowercase → remove URLs/HTML → strip punctuation & digits → remove stopwords → lemmatize.

In [ ]:
STOPWORDS   = set(stopwords.words('english'))
# Keep emotion-bearing negators/intensifiers — removing them hurts accuracy
KEEP_WORDS  = {'not', 'no', 'never', 'nor', 'very', "n't"}
STOPWORDS  -= KEEP_WORDS
LEMMATIZER  = WordNetLemmatizer()

URL_RE      = re.compile(r'https?://\S+|www\.\S+')
HTML_RE     = re.compile(r'<.*?>')
NONALPHA_RE = re.compile(r'[^a-z\s]')

def preprocess(text: str) -> str:
    text = text.lower()
    text = URL_RE.sub(' ', text)
    text = HTML_RE.sub(' ', text)
    text = NONALPHA_RE.sub(' ', text)                  # strip punctuation + digits
    tokens = text.split()
    tokens = [t for t in tokens if t not in STOPWORDS and len(t) > 1]
    tokens = [LEMMATIZER.lemmatize(t) for t in tokens]
    return ' '.join(tokens)

# Apply to all splits
train_df['clean'] = train_df['text'].apply(preprocess)
val_df['clean']   = val_df['text'].apply(preprocess)
test_df['clean']  = test_df['text'].apply(preprocess)

# Before/after preview
sample = train_df.sample(5, random_state=SEED)
for _, row in sample.iterrows():
    print(f"RAW    : {row['text']}")
    print(f"CLEAN  : {row['clean']}")
    print(f"LABEL  : {EMOTION_LABELS[row['label']]}")
    print("-"*70)

## 5 — EDA: class distribution + WordClouds

In [ ]:
# Class distribution
counts = train_df['label'].value_counts().sort_index()
plt.figure(figsize=(9,4))
bars = plt.bar(EMOTION_LABELS, counts.values,
               color=['#4A90E2','#F5A623','#E94B8B','#D0021B','#7B68EE','#50E3C2'])
for b, c in zip(bars, counts.values):
    plt.text(b.get_x()+b.get_width()/2, c+50, str(c), ha='center', fontsize=10)
plt.title('Training Set: Class Distribution')
plt.ylabel('Count'); plt.tight_layout(); plt.show()

# Sentence-length distribution (drives our MAX_LEN choice)
lens = train_df['clean'].str.split().str.len()
print(f"\nClean word-length: mean={lens.mean():.1f}  p95={np.percentile(lens,95):.0f}  "
      f"p99={np.percentile(lens,99):.0f}  max={lens.max()}")

In [ ]:
# Per-emotion WordCloud
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for i, (ax, name) in enumerate(zip(axes.flatten(), EMOTION_LABELS)):
    text = ' '.join(train_df[train_df['label']==i]['clean'].tolist())
    wc = WordCloud(width=400, height=250, background_color='white',
                   max_words=60, colormap='viridis').generate(text)
    ax.imshow(wc, interpolation='bilinear'); ax.axis('off')
    ax.set_title(name, fontsize=14, fontweight='bold')
plt.suptitle('Most common words per emotion (after preprocessing)', fontsize=15)
plt.tight_layout(); plt.show()

## 6 — Step 3: Tokenization & padding

In [ ]:
VOCAB_SIZE = 20_000     # cap on the vocabulary (covers ~99% of words)
MAX_LEN    = 40         # covers p99 of clean sentence length
OOV_TOKEN  = '<OOV>'

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token=OOV_TOKEN)
tokenizer.fit_on_texts(train_df['clean'])              # fit ONLY on train

def to_padded(texts):
    seqs = tokenizer.texts_to_sequences(texts)
    return pad_sequences(seqs, maxlen=MAX_LEN, padding='post', truncating='post')

X_train = to_padded(train_df['clean'])
X_val   = to_padded(val_df['clean'])
X_test  = to_padded(test_df['clean'])

print(f"Vocab size in tokenizer: {len(tokenizer.word_index):,} "
      f"(using top {VOCAB_SIZE:,})")
print(f"Shapes — train {X_train.shape}, val {X_val.shape}, test {X_test.shape}")
print(f"\nExample:")
print("Clean   :", train_df['clean'].iloc[0])
print("Tokens  :", X_train[0][:20], "...")

## 7 — Step 4: Label encoding (integer → one-hot)

In [ ]:
y_train = to_categorical(train_df['label'].values, num_classes=NUM_CLASSES)
y_val   = to_categorical(val_df['label'].values,   num_classes=NUM_CLASSES)
y_test  = to_categorical(test_df['label'].values,  num_classes=NUM_CLASSES)
print("y_train one-hot shape:", y_train.shape)
print("Example label vector :", y_train[0])

## 8 — Step 5: Model architecture (Embedding + BiLSTM + Dense)

In [ ]:
EMBED_DIM   = 128
LSTM_UNITS  = 64
DROPOUT     = 0.3

model = Sequential([
    Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM, input_length=MAX_LEN,
              mask_zero=True, name='embedding'),
    SpatialDropout1D(0.2, name='spatial_dropout'),
    Bidirectional(LSTM(LSTM_UNITS, return_sequences=True, dropout=DROPOUT,
                       recurrent_dropout=0.0), name='bilstm_1'),
    Bidirectional(LSTM(LSTM_UNITS // 2, dropout=DROPOUT,
                       recurrent_dropout=0.0), name='bilstm_2'),
    Dense(64, activation='relu', name='dense'),
    Dropout(DROPOUT, name='dropout'),
    Dense(NUM_CLASSES, activation='softmax', name='output'),
])

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
              loss='categorical_crossentropy',
              metrics=['accuracy'])
model.summary()

## 9 — Step 6: Train

In [ ]:
# Class weights — dair-ai/emotion is imbalanced (Surprise ≈ 3.6%)
from sklearn.utils.class_weight import compute_class_weight
y_train_int = train_df['label'].values
cw_arr = compute_class_weight('balanced', classes=np.unique(y_train_int), y=y_train_int)
class_weight = {int(i): float(w) for i, w in enumerate(cw_arr)}
print("Class weights:", {EMOTION_LABELS[i]: f"{w:.2f}" for i, w in class_weight.items()})

EPOCHS     = 15
BATCH_SIZE = 64

SAVE_DIR = "model/lstm_emotion"
os.makedirs(SAVE_DIR, exist_ok=True)
CKPT_PATH = os.path.join(SAVE_DIR, "best_lstm.h5")

callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=3,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2,
                      min_lr=1e-6, verbose=1),
    ModelCheckpoint(CKPT_PATH, monitor='val_accuracy',
                    save_best_only=True, verbose=1),
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    class_weight=class_weight,
    callbacks=callbacks, verbose=1,
)

### Training curves

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12,4))
ax[0].plot(history.history['accuracy'],     label='train')
ax[0].plot(history.history['val_accuracy'], label='val')
ax[0].set_title('Accuracy'); ax[0].set_xlabel('epoch'); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].plot(history.history['loss'],     label='train')
ax[1].plot(history.history['val_loss'], label='val')
ax[1].set_title('Loss'); ax[1].set_xlabel('epoch'); ax[1].legend(); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 10 — Step 7: Test-set evaluation

In [ ]:
test_probs = model.predict(X_test, verbose=1)
test_preds = np.argmax(test_probs, axis=1)
y_true     = test_df['label'].values

acc = accuracy_score(y_true, test_preds)
mf1 = f1_score(y_true, test_preds, average='macro')
wf1 = f1_score(y_true, test_preds, average='weighted')

print("="*60)
print(f"Test accuracy : {acc*100:.2f}%")
print(f"Macro F1      : {mf1:.4f}")
print(f"Weighted F1   : {wf1:.4f}")
print("="*60)
print(classification_report(y_true, test_preds, target_names=EMOTION_LABELS, digits=3))

In [ ]:
cm = confusion_matrix(y_true, test_preds)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=EMOTION_LABELS, yticklabels=EMOTION_LABELS, cbar=True)
plt.xlabel('Predicted'); plt.ylabel('True')
plt.title('Confusion Matrix — LSTM'); plt.tight_layout(); plt.show()

## 11 — Step 8: Inference helper

In [ ]:
def predict_emotion(text: str, top_k: int = 3):
    """Predict emotion(s) for a raw input sentence."""
    clean = preprocess(text)
    seq = tokenizer.texts_to_sequences([clean])
    padded = pad_sequences(seq, maxlen=MAX_LEN, padding='post', truncating='post')
    probs = model.predict(padded, verbose=0)[0]
    idx = np.argsort(probs)[::-1][:top_k]
    return [(EMOTION_LABELS[i], float(probs[i])) for i in idx]


examples = [
    "I am so happy and grateful for everything in my life!",
    "I miss you so much, you mean the world to me",
    "This makes me really angry and frustrated!",
    "I'm absolutely terrified of what comes next",
    "Wow! I can't believe this just happened to me!",
    "Feeling really down and lost today",
]
for t in examples:
    preds = predict_emotion(t, top_k=3)
    top = preds[0]
    print(f"\n{t!r}")
    print(f"  → {top[0]:<8s} ({top[1]:.2%})")
    print(f"  top-3: " + ", ".join(f"{n}={p:.2f}" for n, p in preds))

## 12 — Save artifacts for the Streamlit app

Saves the trained model (`.keras`) and tokenizer (`.json`) to `model/lstm_emotion/`.
The app's `src/model_loader.py` is set up to auto-detect this directory.

In [ ]:
import json, shutil

# Save model as .h5 (legacy HDF5 — easy to load anywhere)
H5_PATH = os.path.join(SAVE_DIR, 'model.h5')
model.save(H5_PATH)
print(f"Saved model → {H5_PATH}  ({os.path.getsize(H5_PATH)/1024**2:.1f} MB)")

# Save tokenizer + preprocessing config so inference is reproducible
tokenizer_config = {
    'tokenizer_json': tokenizer.to_json(),
    'max_len':       MAX_LEN,
    'vocab_size':    VOCAB_SIZE,
    'oov_token':     OOV_TOKEN,
    'emotion_labels': EMOTION_LABELS,
    'keep_words':    sorted(KEEP_WORDS),
}
TOK_PATH = os.path.join(SAVE_DIR, 'tokenizer.json')
with open(TOK_PATH, 'w') as f:
    json.dump(tokenizer_config, f, indent=2)
print(f"Saved tokenizer → {TOK_PATH}")

print("\nContents of", SAVE_DIR, ":", os.listdir(SAVE_DIR))

# Zip both files for one-click download
zip_path = shutil.make_archive(SAVE_DIR, 'zip', SAVE_DIR)
print(f"\nZipped → {zip_path}  ({os.path.getsize(zip_path)/1024**2:.1f} MB)")
